# Construct BOW, DTM, TFIDF, TFIDF_L2

BOW → DTM → TFIDF → TFIDF_L2, plus the DFIDF computation back-joined to VOCAB.

This is also you declare your bag level and your significant-word selection principle for TFIDF_L2

Note: I have 22 book_ids but 12 true novels. The stories from the collection "poirot investigates" are split into individual book_ids.

I am choosing Chapter as my bag because it is a good narrative WORD that isn't too big...

I am following along with my M07 HW as I complete this section.

## Setup

### Import Libraries

In [1]:
# import libraries
import pandas as pd
import numpy as np
from numpy.linalg import norm
from sklearn.feature_extraction.text import CountVectorizer, TfidfVectorizer, TfidfTransformer

### Configuration

In [2]:
# specify OHCO and bags
OHCO = ['book_id', 'chap_num', 'para_num', 'sent_num', 'token_num']

bags = dict(
    SENTS = OHCO[:4],
    PARAS = OHCO[:3],
    CHAPS = OHCO[:2],
    BOOKS = OHCO[:1]
)

In [3]:
# set chapter as bag
bag = "CHAPS"

### Load Data

In [4]:
# load in tables
CORPUS = pd.read_csv('data/CORPUS.csv', sep='\t').set_index(OHCO)
VOCAB = pd.read_csv('data/VOCAB.csv', sep='\t').set_index('term_str')

## Build BOW

Create a BOW with chapter as the bag.

In [5]:
# insert corpus_to_bow function from M05 Homework
def corpus_to_bow(df=None, bag='CHAPS'):
    '''
    Function that generates a bag of words BOW representation of the CORPUS table.
    Arguments: 
    - Tokens dataframe. Default is entire CORPUS table but can be a subset of it.
    - Choice of bag, i.e. `OHCO` level, such as book, chapter, or paragraph. Default is "CHAPS"
    Output: 
    - BOW table: A multiindex (bag levels + term_str) dataframe and a column count 'n'
    '''
    if df is None:
        raise ValueError("Must pass a DataFrame")
    
    # grab column names for bag
    bag_cols = bags[bag]
    
    BOW = df.groupby(bag_cols + ['term_str']).term_str.count().to_frame('n')

    return BOW

In [6]:
# create BOW with chapter as bag
BOW_chaps = corpus_to_bow(CORPUS, bag)

BOW_chaps

n
book_id                      chap_num term_str      
giants-bread                 0        a           50
                                      about        3
                                      above        1
                                      absolute     1
                                      absolutely   1
...                                               ..
the-tragedy-at-marsdon-manor 1        yielded      1
                                      you         77
                                      young       11
                                      your        10
                                      yourselfit   1

[256057 rows x 1 columns]

## Build DTM

Unstack the BOW into DTM and replace nulls with 0s.

In [7]:
# unstack BOW into DTM and replace nulls with 0s
DTM = BOW_chaps.unstack(fill_value=0)

# drop the 'n' top level column artifact that came over from BOW count
DTM.columns = DTM.columns.droplevel(0)

DTM

term_str                               1  10  100  100000  1015  1019  102  \
book_id                      chap_num                                        
giants-bread                 0         0   0    0       0     0     0    0   
                             1         1   0    0       0     0     0    0   
                             2         0   0    0       0     0     0    0   
                             3         0   0    0       0     0     0    0   
                             4         0   0    0       0     0     0    0   
...                                   ..  ..  ...     ...   ...   ...  ...   
the-seven-dials-mystery      31        0   0    0       0     0     0    0   
                             32        2   0    0       0     0     0    0   
                             33        0   0    0       0     0     0    0   
                             34        0   0    0       0     0     0    0   
the-tragedy-at-marsdon-manor 1         0   0    0       0     0     0    0   

term_str                               1023  103  1030  ...  éliseif  élysées  \
book_id                      chap_num                   ...                     
giants-bread                 0            0    0     0  ...        0        0   
                             1            0    0     0  ...        0        0   
                             2            0    0     0  ...        0        0   
                             3            0    0     0  ...        0        0   
                             4            0    0     0  ...        0        0   
...                                     ...  ...   ...  ...      ...      ...   
the-seven-dials-mystery      31           0    0     0  ...        0        0   
                             32           0    0     0  ...        0        0   
                             33           0    0     0  ...        0        0   
                             34           0    0     0  ...        0        0   
the-tragedy-at-marsdon-manor 1            0    0     0  ...        0        0   

term_str                               émigrés  épatant  épatantsthese  \
book_id                      chap_num                                    
giants-bread                 0               0        0              0   
                             1               0        0              0   
                             2               0        0              0   
                             3               0        0              0   
                             4               0        0              0   
...                                        ...      ...            ...   
the-seven-dials-mystery      31              0        0              0   
                             32              0        0              0   
                             33              0        0              0   
                             34              0        0              0   
the-tragedy-at-marsdon-manor 1               0        0              0   

term_str                               épouvantable  évidemment  évident  \
book_id                      chap_num                                      
giants-bread                 0                    0           0        0   
                             1                    0           0        0   
                             2                    0           0        0   
                             3                    0           0        0   
                             4                    0           0        0   
...                                             ...         ...      ...   
the-seven-dials-mystery      31                   0           0        0   
                             32                   0           0        0   
                             33                   0           0        0   
                             34                   0           0        0   
the-tragedy-at-marsdon-manor 1                    0     

## Build TFIDF

Use SKLearn to convert DTM into a TFIDF data frame by passing DTM to TfidfTransformer with defaults and then saving the results to a dataframe TFIDF that preserves the index and columns of DTM.

In [8]:
# instantiate tfidf engine with norm=None to get raw TFIDF
tfidf_engine = TfidfTransformer(norm=None)

# fit and transform the DTM and then convert output back into a dataframe
TFIDF = pd.DataFrame(
    tfidf_engine.fit_transform(DTM).toarray(), #TfidfTransformer() ouputs a sparse matrix so we use .toarray() to convert it to a dense one
    index = DTM.index,
    columns = DTM.columns
)

TFIDF

term_str                                      1   10  100  100000  1015  1019  \
book_id                      chap_num                                           
giants-bread                 0         0.000000  0.0  0.0     0.0   0.0   0.0   
                             1         3.608844  0.0  0.0     0.0   0.0   0.0   
                             2         0.000000  0.0  0.0     0.0   0.0   0.0   
                             3         0.000000  0.0  0.0     0.0   0.0   0.0   
                             4         0.000000  0.0  0.0     0.0   0.0   0.0   
...                                         ...  ...  ...     ...   ...   ...   
the-seven-dials-mystery      31        0.000000  0.0  0.0     0.0   0.0   0.0   
                             32        7.217687  0.0  0.0     0.0   0.0   0.0   
                             33        0.000000  0.0  0.0     0.0   0.0   0.0   
                             34        0.000000  0.0  0.0     0.0   0.0   0.0   
the-tragedy-at-marsdon-manor 1         0.000000  0.0  0.0     0.0   0.0   0.0   

term_str                               102  1023  103  1030  ...  éliseif  \
book_id                      chap_num                        ...            
giants-bread                 0         0.0   0.0  0.0   0.0  ...      0.0   
                             1         0.0   0.0  0.0   0.0  ...      0.0   
                             2         0.0   0.0  0.0   0.0  ...      0.0   
                             3         0.0   0.0  0.0   0.0  ...      0.0   
                             4         0.0   0.0  0.0   0.0  ...      0.0   
...                                    ...   ...  ...   ...  ...      ...   
the-seven-dials-mystery      31        0.0   0.0  0.0   0.0  ...      0.0   
                             32        0.0   0.0  0.0   0.0  ...      0.0   
                             33        0.0   0.0  0.0   0.0  ...      0.0   
                             34        0.0   0.0  0.0   0.0  ...      0.0   
the-tragedy-at-marsdon-manor 1         0.0   0.0  0.0   0.0  ...      0.0   

term_str                               élysées  émigrés  épatant  \
book_id                      chap_num                              
giants-bread                 0             0.0      0.0      0.0   
                             1             0.0      0.0      0.0   
                             2             0.0      0.0      0.0   
                             3             0.0      0.0      0.0   
                             4             0.0      0.0      0.0   
...                                        ...      ...      ...   
the-seven-dials-mystery      31            0.0      0.0      0.0   
                             32            0.0      0.0      0.0   
                             33            0.0      0.0      0.0   
                             34            0.0      0.0      0.0   
the-tragedy-at-marsdon-manor 1             0.0      0.0      0.0   

term_str                               épatantsthese  épouvantable  \
book_id                      chap_num                                
giants-bread                 0                   0.0           0.0   
                             1                   0.0           0.0   
                             2                   0.0           0.0   
                             3                   0.0           0.0   
                             4                   0.0           0.0   
...                                              ...           ...   
the-seven-dials-mystery      31                  0.0           0.0   
                             32                  0.0           0.0   
                             33                  0.0           0.0   
                             34                  0.0           0.0   
the-tragedy-at-marsdon-manor 1                   0.0           0.0   

term_str                               évidemment  évident  êtes  ⁷₁₁  
book_id                      chap_num                                  
gian

## Reduce Feature Space (and compute DFIDF)

Reduce the feature space of DTM, i.e. the vocabulary, by filtering for words whose document frequency is greater than or equal to 50 and information is greater than or equal to 10.

I have chosen these same thresholds as the M07 HW to start with because there were 320 documents in that corpus and I have 325 in mine.

In [9]:
# calculate DF (document frequency)
df_count = (DTM > 0).sum()

# calculate I (information)
tf = DTM.sum()
p = tf / tf.sum()
information = -np.log2(p)

In [10]:
# create term stats dataframe
TERM_STATS = pd.DataFrame({'df': df_count, 'i': information})

TERM_STATS

,df,i
term_str,,
1,23,14.735375
10,2,18.689571
100,2,18.689571
100000,1,19.689571
1015,1,19.689571
...,...,...
épouvantable,1,19.689571
évidemment,3,18.104609
évident,1,19.689571


In [11]:
# add stop words bool
TERM_STATS['stop'] = VOCAB['stop']

In [12]:
# get index to filter for df_count >= 50 and i >= 10 and remove stop words
keep_terms = TERM_STATS[
    (TERM_STATS['df'] >= 50) & 
    (TERM_STATS['i'] >= 10) & 
    ~TERM_STATS['stop']
].index

keep_terms

Index(['able', 'abruptly', 'absolutely', 'absurd', 'accepted', 'account',
       'across', 'act', 'actually', 'added',
       ...
       'writing', 'written', 'wrong', 'wrote', 'yard', 'year', 'years',
       'yesterday', 'yet', 'young'],
      dtype='str', name='term_str', length=907)

In [13]:
# apply filter to TFIDF table
TFIDF_reduced = TFIDF[keep_terms]

TFIDF_reduced

term_str                                   able  abruptly  absolutely  \
book_id                      chap_num                                   
giants-bread                 0         0.000000  0.000000    2.243603   
                             1         0.000000  0.000000    0.000000   
                             2         0.000000  0.000000    0.000000   
                             3         0.000000  0.000000    0.000000   
                             4         3.552524  2.779564    0.000000   
...                                         ...       ...         ...   
the-seven-dials-mystery      31        1.776262  0.000000    2.243603   
                             32        0.000000  2.779564    0.000000   
                             33        5.328786  0.000000    2.243603   
                             34        0.000000  0.000000    0.000000   
the-tragedy-at-marsdon-manor 1         1.776262  0.000000    4.487205   

term_str                                 absurd  accepted  account    across  \
book_id                      chap_num                                          
giants-bread                 0         2.567390  0.000000      0.0  1.668904   
                             1         0.000000  2.835654      0.0  1.668904   
                             2         0.000000  2.835654      0.0  3.337807   
                             3         0.000000  0.000000      0.0  0.000000   
                             4         5.134779  2.835654      0.0  0.000000   
...                                         ...       ...      ...       ...   
the-seven-dials-mystery      31        0.000000  0.000000      0.0  0.000000   
                             32        0.000000  0.000000      0.0  1.668904   
                             33        0.000000  0.000000      0.0  0.000000   
                             34        0.000000  0.000000      0.0  0.000000   
the-tragedy-at-marsdon-manor 1         0.000000  2.835654      0.0  1.668904   

term_str                                    act  actually     added  ...  \
book_id                      chap_num                                ...   
giants-bread                 0         0.000000  0.000000  0.000000  ...   
                             1         0.000000  0.000000  0.000000  ...   
                             2         0.000000  0.000000  0.000000  ...   
                             3         0.000000  0.000000  0.000000  ...   
                             4         0.000000  2.320989  2.142506  ...   
...                                         ...       ...       ...  ...   
the-seven-dials-mystery      31        0.000000  0.000000  0.000000  ...   
                             32        0.000000  0.000000  0.000000  ...   
                             33        0.000000  9.283957  2.142506  ...   
                             34        0.000000  2.320989  0.000000  ...   
the-tragedy-at-marsdon-manor 1         5.671307  0.000000  2.142506  ...   

term_str                               writing  written     wrong     wrote  \
book_id                      chap_num                                         
giants-bread                 0         0.00000      0.0  1.810164  0.000000   
                             1         0.00000      0.0  0.000000  0.000000   
                             2         0.00000      0.0  3.620327  0.000000   
                             3         0.00000      0.0  0.000000  0.000000   
                             4         0.00000      0.0  0.000000  0.000000   
...                                        ...      ...       ...       ...   
the-seven-dials-mystery      31        0.00000      0.0  0.000000  0.000000   
                             32        0.00000      0.0  0.000000  0.000000   
                             33        0.00000      0.0  1.810164  2.496438   
                             34        5.41872      0.0  0.000000  0.000000   
the-tragedy-at-marsdon-manor 1         0.00000      0.0  0.000000  0.000

## Normalize (TFIDF_L2)

L2 normalize TFIDF_reduced. 

L2 normalization is applied after feature space reduction so that row vector norms reflect only the significant term dimensions that PCA will operate on.

To normalize a vector, we divide each element by the length of its norm $L_p$. Length measures vary by $p$. In the case of L2 normaliztion, $L_2$ is the square root of the sum of each element squared.

In [14]:
TFIDF_L2 = TFIDF_reduced.apply(lambda x: x / norm(x), axis=1) # Pythagorean, AKA Euclidean

TFIDF_L2

term_str                                   able  abruptly  absolutely  \
book_id                      chap_num                                   
giants-bread                 0         0.000000  0.000000    0.050124   
                             1         0.000000  0.000000    0.000000   
                             2         0.000000  0.000000    0.000000   
                             3         0.000000  0.000000    0.000000   
                             4         0.033423  0.026151    0.000000   
...                                         ...       ...         ...   
the-seven-dials-mystery      31        0.012581  0.000000    0.015891   
                             32        0.000000  0.060086    0.000000   
                             33        0.057234  0.000000    0.024098   
                             34        0.000000  0.000000    0.000000   
the-tragedy-at-marsdon-manor 1         0.015902  0.000000    0.040171   

term_str                                 absurd  accepted  account    across  \
book_id                      chap_num                                          
giants-bread                 0         0.057357  0.000000      0.0  0.037285   
                             1         0.000000  0.037165      0.0  0.021873   
                             2         0.000000  0.057522      0.0  0.067709   
                             3         0.000000  0.000000      0.0  0.000000   
                             4         0.048309  0.026678      0.0  0.000000   
...                                         ...       ...      ...       ...   
the-seven-dials-mystery      31        0.000000  0.000000      0.0  0.000000   
                             32        0.000000  0.000000      0.0  0.036077   
                             33        0.000000  0.000000      0.0  0.000000   
                             34        0.000000  0.000000      0.0  0.000000   
the-tragedy-at-marsdon-manor 1         0.000000  0.025386      0.0  0.014941   

term_str                                    act  actually     added  ...  \
book_id                      chap_num                                ...   
giants-bread                 0         0.000000  0.000000  0.000000  ...   
                             1         0.000000  0.000000  0.000000  ...   
                             2         0.000000  0.000000  0.000000  ...   
                             3         0.000000  0.000000  0.000000  ...   
                             4         0.000000  0.021836  0.020157  ...   
...                                         ...       ...       ...  ...   
the-seven-dials-mystery      31        0.000000  0.000000  0.000000  ...   
                             32        0.000000  0.000000  0.000000  ...   
                             33        0.000000  0.099715  0.023012  ...   
                             34        0.000000  0.061549  0.000000  ...   
the-tragedy-at-marsdon-manor 1         0.050772  0.000000  0.019181  ...   

term_str                                writing  written     wrong     wrote  \
book_id                      chap_num                                          
giants-bread                 0         0.000000      0.0  0.040440  0.000000   
                             1         0.000000      0.0  0.000000  0.000000   
                             2         0.000000      0.0  0.073440  0.000000   
                             3         0.000000      0.0  0.000000  0.000000   
                             4         0.000000      0.0  0.000000  0.000000   
...                                         ...      ...       ...       ...   
the-seven-dials-mystery      31        0.000000      0.0  0.000000  0.000000   
                             32        0.000000      0.0  0.000000  0.000000   
                             33        0.000000      0.0  0.019442  0.026813   
                             34        0.143697      0.0  0.000000  0.000000   
the-tragedy-at-marsdon-manor 1         0.000000      0.0  0.

## Update VOCAB with DFIDF

In [15]:
# add df column to VOCAB
VOCAB['df'] = TERM_STATS['df'] # bring over df (and only df) from TERM_STATS

# compute and add idf to VOCAB (recall idf=log2(N/df) where N is number of bags)
N = CORPUS.groupby(bags[bag]).ngroups
VOCAB['idf'] = np.log2(N/VOCAB['df'])

# compute and add dfidf VOCAB
VOCAB['dfidf'] = VOCAB['df'] * VOCAB['idf'] # DFIDF is the same as mean boolean TFIDF (doc freq * inv doc freq)

# VOCAB

### Top 20 Significant Words By DFIDF

In [16]:
VOCAB.sort_values('dfidf', ascending=False).head(20)

,n,n_chars,p,i,max_pos,max_pos_group,lemma,n_pos_group,n_pos,stop,porter_stem,df,idf,dfidf
term_str,,,,,,,,,,,,,,
hall,240,4,0.000284,11.782681,NN,NN,hall,1,2,False,hall,120,1.437405,172.488637
anyway,175,6,0.000207,12.238360,RB,RB,anyway,3,3,False,anyway,120,1.437405,172.488637
ten,209,3,0.000247,11.982212,CD,CD,ten,2,2,False,ten,120,1.437405,172.488637
silence,179,7,0.000212,12.205756,NN,NN,silence,3,3,False,silenc,120,1.437405,172.488637
letter,324,6,0.000383,11.349721,NN,NN,letter,1,1,False,letter,120,1.437405,172.488637
suggested,169,9,0.000200,12.288692,VBD,VB,suggest,1,2,False,suggest,119,1.449478,172.487899
run,177,3,0.000209,12.221966,VB,VB,run,2,5,False,run,119,1.449478,172.487899
surprised,154,9,0.000182,12.422785,JJ,JJ,surprised,2,3,False,surpris,119,1.449478,172.487899
lay,169,3,0.000200,12.288692,VBD,VB,lie,2,4,False,lay,119,1.449478,172.487899


## Save Outputs

In [17]:
# save the updated VOCAB table to csv (replace the existing one created by 03-VOCAB.ipynb)
VOCAB.to_csv('data/VOCAB.csv', sep='\t', index=True)

# save the BOW table to csv
BOW_chaps.to_csv('data/BOW_chaps.csv', sep='\t', index=True)

# save the DTM table to csv
DTM.to_csv('data/DTM.csv', sep='\t', index=True)

# save the TFIDF table to csv
TFIDF.to_csv('data/TFIDF.csv', sep='\t', index=True)

# save the TFIDF_L2 table to csv
TFIDF_L2.to_csv('data/TFIDF_L2.csv', sep='\t', index=True)